# Задача 3. K-th

**Объяснение решения**

Ищем `k`-й элемент по убыванию. Повторы учитываются: два одинаковых числа занимают две позиции. В порядке возрастания нужному элементу соответствует индекс `target = len(nums) - k`.

Используем quickselect. Берем опорный элемент из середины текущего участка и делим участок на три части: меньше опорного, равные ему и больше. Равные занимают индексы от `low` до `high`.

- Если `target < low`, продолжаем слева.
- Если `target > high`, продолжаем справа.
- Иначе возвращаем элемент на позиции `target`.

Все значения слева меньше опорного, а справа больше. Поэтому ответ может находиться только в выбранной части. Внутри самих частей порядок не важен.

В разбиении `i` — текущий индекс, `low` — место для следующего меньшего элемента, `high` — для следующего большего. После обмена с `high` не увеличиваем `i`: пришедшее справа число еще нужно проверить.

Работаем с копией массива. Для пустого массива или `k` вне диапазона от `1` до длины массива выбрасываем `ValueError`.

**Решение**

In [1]:
def quickselect(nums, k):
    if not isinstance(k, int):
        raise TypeError("k должно быть целым числом")
    if not 1 <= k <= len(nums):
        raise ValueError("k должно быть от 1 до длины массива")

    values = nums.copy()
    target = len(values) - k
    left, right = 0, len(values) - 1

    while left <= right:
        pivot = values[(left + right) // 2]
        low = i = left
        high = right

        while i <= high:
            if values[i] < pivot:
                values[low], values[i] = values[i], values[low]
                low += 1
                i += 1
            elif values[i] > pivot:
                values[i], values[high] = values[high], values[i]
                high -= 1
            else:
                i += 1

        if target < low:
            right = low - 1
        elif target > high:
            left = high + 1
        else:
            return values[target]

**Пример**

In [2]:
print(quickselect([3, 2, 1, 5, 6, 4], 2))
print(quickselect([3, 2, 3, 1, 2, 4, 5, 5, 6], 4))

5
4


Для пошагового разбора возьмем `[8, -3, 5, 8, 0, 12, -7, 5, 2, 10, 1]`, `k = 5`.

Всего `11` элементов, поэтому `target = 11 - 5 = 6`. Индексы считаем с нуля, границы участков включены.

In [3]:
nums = [8, -3, 5, 8, 0, 12, -7, 5, 2, 10, 1]
k = 5

print("Исходный массив:", nums)
print("k:", k)
print("Ответ:", quickselect(nums, k))

Исходный массив: [8, -3, 5, 8, 0, 12, -7, 5, 2, 10, 1]
k: 5
Ответ: 5


После каждого разбиения получаем следующие состояния копии массива:

| Шаг | Участок | Опорный | Массив после разбиения | Индексы равных опорному |
|---|---|---|---|---|
| 1 | `(0, 10)` | `12` | `[8, -3, 5, 8, 0, -7, 5, 2, 10, 1, 12]` | `(10, 10)` |
| 2 | `(0, 9)` | `0` | `[-7, -3, 0, 8, 5, 5, 2, 10, 1, 8, 12]` | `(2, 2)` |
| 3 | `(3, 9)` | `2` | `[-7, -3, 0, 1, 2, 5, 10, 5, 8, 8, 12]` | `(4, 4)` |
| 4 | `(5, 9)` | `5` | `[-7, -3, 0, 1, 2, 5, 5, 8, 8, 10, 12]` | `(5, 6)` |

1. `6 < 10` — оставляем участок `(0, 9)`.
2. `6 > 2` — оставляем участок `(3, 9)`.
3. `6 > 4` — оставляем участок `(5, 9)`.
4. Индекс `6` попал в диапазон `(5, 6)`, где оба элемента равны `5`. Возвращаем `5`.

Пять наибольших значений здесь — `12`, `10`, `8`, `8`, `5`.

**Сложность**

Пусть `n` — длина массива, сравнение чисел занимает `O(1)`.

| Случай | Время | Дополнительная память |
|---|---|---|
| В среднем при случайном порядке элементов | `O(n)` | `O(n)` |
| Худший случай | `O(n²)` | `O(n)` |
| Все элементы равны | `O(n)` | `O(n)` |

Одно разбиение занимает время, линейное по длине участка. Продолжаем только в одной части. При равномерных разбиениях работа складывается как `n + n/2 + n/4 + ...`, что дает `O(n)`.

Элемент из середины участка не обязательно является медианой. В худшем случае каждый раз убираем только один элемент: `n + (n - 1) + ... + 1 = O(n²)`.

Если все числа равны, ответ находится после первого разбиения. Копия массива занимает `O(n)` памяти, а индексы и опорный элемент — `O(1)`. Рекурсивного стека нет.

**Тесты**

`sorted` используется только в тестах как эталон для проверки результата.

In [4]:
import random
import unittest
from itertools import product


class TestQuickselect(unittest.TestCase):
    def check(self, nums, ranks=None):
        before = nums.copy()
        expected = sorted(nums, reverse=True)
        if ranks is None:
            ranks = range(1, len(nums) + 1)

        for k in ranks:
            with self.subTest(size=len(nums), k=k):
                self.assertEqual(quickselect(nums, k), expected[k - 1])
                self.assertEqual(nums, before)

    def test_first_example(self):
        self.assertEqual(quickselect([3, 2, 1, 5, 6, 4], 2), 5)

    def test_second_example(self):
        self.assertEqual(quickselect([3, 2, 3, 1, 2, 4, 5, 5, 6], 4), 4)

    def test_empty(self):
        with self.assertRaises(ValueError):
            quickselect([], 1)

    def test_single(self):
        self.check([7])

    def test_two_elements(self):
        for nums in ([1, 2], [2, 1], [2, 2]):
            self.check(nums)

    def test_visualization_example(self):
        self.check([8, -3, 5, 8, 0, 12, -7, 5, 2, 10, 1])

    def test_duplicates(self):
        self.check([5, 5, 4, 4, 4, 3, 2, 2, 1])

    def test_all_equal(self):
        self.check([5] * 10000, (1, 5000, 10000))

    def test_negative_and_zero(self):
        self.check([0, -9, -1, 0, -5, -9])

    def test_sorted(self):
        self.check(list(range(5000)), (1, 2500, 5000))

    def test_reversed(self):
        self.check(list(range(4999, -1, -1)), (1, 2500, 5000))

    def test_large_numbers(self):
        self.check([10**100, 0, -10**100, 7, -7])

    def test_floats(self):
        self.check([1.5, -0.5, 2.0, 0.0, 1.5])

    def test_random(self):
        rng = random.Random(42)
        for size in (3, 10, 100, 1000, 5000):
            nums = [rng.randint(-1000, 1000) for _ in range(size)]
            ranks = {1, size, (size + 1) // 2, rng.randint(1, size)}
            self.check(nums, ranks)

    def test_all_small_arrays(self):
        for size in range(1, 7):
            for values in product((-1, 0, 1), repeat=size):
                self.check(list(values))

    def test_invalid_k(self):
        for k in (0, -1, 4):
            with self.subTest(k=k):
                with self.assertRaises(ValueError):
                    quickselect([3, 1, 2], k)

    def test_invalid_type(self):
        for k in (1.5, "2", None):
            with self.subTest(k=k):
                with self.assertRaises(TypeError):
                    quickselect([3, 1, 2], k)

In [5]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestQuickselect)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Некоторые тесты завершились ошибкой"
print(f"Успешно пройдено тестов: {test_result.testsRun}")

test_all_equal (__main__.TestQuickselect.test_all_equal) ... ok
test_all_small_arrays (__main__.TestQuickselect.test_all_small_arrays) ... ok
test_duplicates (__main__.TestQuickselect.test_duplicates) ... ok
test_empty (__main__.TestQuickselect.test_empty) ... ok
test_first_example (__main__.TestQuickselect.test_first_example) ... ok
test_floats (__main__.TestQuickselect.test_floats) ... ok
test_invalid_k (__main__.TestQuickselect.test_invalid_k) ... ok
test_invalid_type (__main__.TestQuickselect.test_invalid_type) ... ok
test_large_numbers (__main__.TestQuickselect.test_large_numbers) ... ok
test_negative_and_zero (__main__.TestQuickselect.test_negative_and_zero) ... ok
test_random (__main__.TestQuickselect.test_random) ... ok
test_reversed (__main__.TestQuickselect.test_reversed) ... ok
test_second_example (__main__.TestQuickselect.test_second_example) ... ok
test_single (__main__.TestQuickselect.test_single) ... ok
test_sorted (__main__.TestQuickselect.test_sorted) ... ok
test_two_e

Успешно пройдено тестов: 17
